# Preparación del corpus NLP para proyecto OSINT

Este notebook construye `data/dataset_nlp.csv` a partir de las fuentes textuales autorizadas y genera un EDA inicial con gráficas guardadas en `outputs/figures/`.

Se excluyen deliberadamente `nasa_firms.csv` y `opensky.csv`, porque son datos geoespaciales/operacionales y no pertenecen al corpus textual principal.

In [1]:
from __future__ import annotations

import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set(style="whitegrid")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/figures")
INPUT_FILES = [
    DATA_DIR / "bbc_rss.csv",
    DATA_DIR / "aljazeera_rss.csv",
    DATA_DIR / "google_news_rss.csv",
    DATA_DIR / "gdelt.csv",
]
OUTPUT_CSV = DATA_DIR / "dataset_nlp.csv"
REQUIRED_COLUMNS = ["timestamp", "source", "title", "text", "url"]
STOPWORDS = {
    "the", "and", "for", "with", "that", "this",
    "from", "have", "has", "are", "was", "were",
    "been", "their", "they", "but", "not", "will",
    "its", "his", "her", "she", "he", "you",
    "your", "them", "about", "than", "who", "which",
    "when", "where", "what", "can", "all", "any",
    "one", "our", "there", "more", "also", "into",
    "after", "over", "such", "had", "may", "many",
    "most", "other", "even"
}

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
def normalize_string(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFC", text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def load_source_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["timestamp"])
    missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas requeridas en {path}: {missing_columns}")
    return df.copy()

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df[REQUIRED_COLUMNS]
    for col in ["source", "title", "text", "url"]:
        df[col] = df[col].map(normalize_string)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df = df.dropna(subset=["timestamp", "source", "title", "text", "url"])
    df["full_text"] = (df["title"] + " " + df["text"]).str.strip().map(normalize_string)
    df = df[df["full_text"] != ""]
    df = df.drop_duplicates(subset=["timestamp", "source", "title", "text", "url", "full_text"]).reset_index(drop=True)
    return df

In [ ]:
datasets = []
missing_summary = []
for path in INPUT_FILES:
    df = load_source_csv(path)
    missing_summary.append(df[REQUIRED_COLUMNS].isna().sum())
    cleaned = clean_dataframe(df)
    datasets.append(cleaned)

corpus = pd.concat(datasets, ignore_index=True)
corpus = corpus.sort_values("timestamp").reset_index(drop=True)
duplicates_removed = corpus.duplicated(subset=["timestamp", "source", "title", "text", "url", "full_text"]).sum()
corpus = corpus.drop_duplicates(subset=["timestamp", "source", "title", "text", "url", "full_text"]).reset_index(drop=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
corpus.to_csv(OUTPUT_CSV, index=False)
print("Generado data/dataset_nlp.csv con filas: {}".format(len(corpus)))
print("Duplicados finales eliminados: {}".format(int(duplicates_removed)))
print("Fuentes integradas: {}".format(corpus["source"].value_counts().to_dict()))
print("Longitud promedio full_text: {:.1f}".format(corpus["full_text"].str.len().mean()))

In [ ]:
def plot_distribution_by_source(df: pd.DataFrame) -> None:
    counts = df["source"].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x=counts.values, y=counts.index, palette="muted", ax=ax)
    ax.set_title("Distribución de filas por fuente")
    ax.set_xlabel("Cantidad de registros")
    ax.set_ylabel("Fuente")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "distribution_by_source.png", dpi=200)
    plt.close(fig)

def plot_text_length_distribution(df: pd.DataFrame) -> None:
    df = df.copy()
    df["text_length"] = df["full_text"].str.len()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(df["text_length"], bins=40, kde=True, color="#2c7fb8", ax=ax)
    ax.set_title("Distribución de longitud de textos (full_text)")
    ax.set_xlabel("Longitud de texto (caracteres)")
    ax.set_ylabel("Número de registros")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "text_length_distribution.png", dpi=200)
    plt.close(fig)

def plot_timeline(df: pd.DataFrame) -> None:
    df = df.copy()
    df["date"] = df["timestamp"].dt.date
    timeline = df.groupby(["date", "source"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(12, 5))
    timeline.plot(kind="line", marker="o", ax=ax)
    ax.set_title("Timeline temporal por fuente")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Número de registros")
    ax.legend(title="Fuente")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "timeline_per_source.png", dpi=200)
    plt.close(fig)

def plot_missing_values(df: pd.DataFrame) -> None:
    missing = df[REQUIRED_COLUMNS].isna().sum()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=missing.values, y=missing.index, palette="Blues_d", ax=ax)
    ax.set_title("Valores faltantes por columna antes de limpieza")
    ax.set_xlabel("Cantidad de valores faltantes")
    ax.set_ylabel("Columna")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "missing_values_before_cleaning.png", dpi=200)
    plt.close(fig)

def plot_top_words(df: pd.DataFrame, top_n: int = 20) -> None:
    all_text = " ".join(df["full_text"].astype(str).tolist()).lower()
    tokens = re.findall(r"\b[\w']{2,}\b", all_text)
    tokens = [token for token in tokens if token not in STOPWORDS]
    counter = Counter(tokens)
    most_common = counter.most_common(top_n)
    words, counts = zip(*most_common)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=list(counts), y=list(words), palette="viridis", ax=ax)
    ax.set_title("Palabras más frecuentes en el corpus NLP")
    ax.set_xlabel("Frecuencia")
    ax.set_ylabel("Palabra")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "top_words.png", dpi=200)
    plt.close(fig)

plot_distribution_by_source(corpus)
plot_text_length_distribution(corpus)
plot_timeline(corpus)
plot_missing_values(pd.concat(missing_summary, axis=1).sum(axis=1))
plot_top_words(corpus)
print("Gráficas guardadas en ../outputs/figures/")
